# Session 4 - Indexes and Query Optimization Basics

Welcome. In this session we will learn:

- what an **index** is for (and when it helps)
- how to read **`EXPLAIN`**
- how **`EXPLAIN ANALYZE`** shows real timings
- **sargable** filters (filters that can use an index cleanly)

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - What is an index?

## Phone-book story

Finding "Sharma" in a phone book:

```
No index (table scan)              With an index (lookup)
----------------------             ----------------------
Open page 1, scan names            Open index: S -> page 412
Open page 2, scan names            Go straight to page 412
... keep going ...                 Done
```

MySQL stores table rows on disk. An **index** is a sorted shortcut (usually a B-tree) that points to matching rows.

## Diagram: table vs index

```
training.orders (table heap / data)     idx on customer_id (sorted shortcut)
+-----------+--------+--------+         +-------------+-----------+
| order_id  | cust   | date   |         | customer_id | -> row    |
+-----------+--------+--------+         +-------------+-----------+
| 10248     | 1001   | 2023.. |         | 1001        | -> 10248  |
| 10249     | 2044   | 2023.. |         | 1001        | -> 11002  |
| 10250     | 1001   | 2023.. |         | 1002        | -> 10255  |
+-----------+--------+--------+         +-------------+-----------+

Query: WHERE customer_id = 1001
  -> use index, fetch only matching order rows
```

## Trade-off (keep this small)

| Upside | Cost |
|--------|------|
| Faster filters / joins on that column | Insert/update/delete must update the index |
| Less data read when selective | Uses disk space |

**When indexes help**

| Filter style | Typical plan | Why |
|--------------|--------------|-----|
| Selective (`customer_id = 1001`) | Index lookup | few matching rows |
| Broad (`freight > 50`) | Often full scan | too many rows still match |

Selective filters love indexes. Broad filters may still scan the table - and that can be the right choice.


---
# Chapter 2 - Indexes already on these tables

Useful indexes were created when the lab database was built (keys like `customer_id`, `order_date`, and foreign keys).

We **feel** them through `EXPLAIN` / `EXPLAIN ANALYZE` next - no need to invent new permanent indexes on `training.*`.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Indexes</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Reading EXPLAIN

`EXPLAIN` shows the **plan** MySQL chooses - it does **not** run the full query just to time it.

Think of it as asking the GPS for the route before you drive.

## Columns worth reading first

| Column | Simple meaning |
|--------|----------------|
| `type` | How rows are accessed (`const`, `ref`, `range`, `index`, `ALL`, ...) |
| `key` | Which index was chosen (`NULL` = no index used) |
| `rows` | Rough estimate of rows MySQL expects to examine |
| `Extra` | Notes like `Using where`, `Using index` |

## Access types (short map)

```
const / ref / range  ->  usually "used an index / tight access"
index                ->  walked an index (not always bad)
ALL                  ->  read the table (full scan)
```

None of these is always wrong. A full scan can be fine for small tables or broad filters.


### Lookup by primary key (expect a tight plan)


**Story:** Look up one customer by id - like opening one locker with the exact key.


In [1]:
EXPLAIN
SELECT *
FROM training.customers
WHERE customer_id = 1001;


+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
| id | select_type | table     | partitions | type  | possible_keys | key     | key_len | ref   | rows | filtered | Extra |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | customers | NULL       | const | PRIMARY       | PRIMARY | 4       | const |    1 |   100.00 | NULL  |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+


### Filter on an indexed date column


**Use case:** "Show me orders on one date." An index on `order_date` can jump to that date.


In [2]:
EXPLAIN
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
| id | select_type | table  | partitions | type | possible_keys         | key                   | key_len | ref   | rows | filtered | Extra |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | orders | NULL       | ref  | idx_orders_order_date | idx_orders_order_date | 3       | const |   57 |   100.00 | NULL  |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+


### Broad filter (often a table scan)


**Story:** "Show freight over 50" may match a huge share of orders.
Scanning the table can be cheaper than bouncing through an index for almost every row.


In [3]:
EXPLAIN
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
| id | select_type | table  | partitions | type | possible_keys | key  | key_len | ref  | rows  | filtered | Extra       |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
|  1 | SIMPLE      | orders | NULL       | ALL  | NULL          | NULL | NULL    | NULL | 38996 |    33.33 | Using where |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+


### What to look for in the plan

1. Does `key` show an index name, or `NULL`?
2. Is `type` a tight access (`const` / `ref` / `range`) or `ALL`?
3. Is `rows` small or huge?

Compare the three plans above: primary-key lookup, indexed date, broad freight filter.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - EXPLAIN</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 4 - EXPLAIN ANALYZE (timed run)

`EXPLAIN` = "show the plan."  
`EXPLAIN ANALYZE` = "run the query and show the plan **with real timings / row counts**."

```
EXPLAIN            ->  estimated route on the map
EXPLAIN ANALYZE    ->  drive the route, then report how long each turn took
```

Use `EXPLAIN` while learning the shape of a plan.  
Use `EXPLAIN ANALYZE` when you want evidence about cost on **your** data.

> Needs MySQL 8.0.18+. Most current installs support it.


### EXPLAIN ANALYZE for a primary-key lookup


In [4]:
EXPLAIN ANALYZE
SELECT *
FROM training.customers
WHERE customer_id = 1001;


-> Rows fetched before execution  (cost=0..0 rows=1) (actual time=345e-6..380e-6 rows=1 loops=1)


### EXPLAIN ANALYZE for an indexed date filter


In [5]:
EXPLAIN ANALYZE
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


-> Index lookup on orders using idx_orders_order_date (order_date=DATE'2023-06-15')  (cost=19.9 rows=57) (actual time=0.0525..0.0936 rows=57 loops=1)


### EXPLAIN ANALYZE for a broad freight filter


Compare the actual time / rows examined here with the tight lookups above.


In [6]:
EXPLAIN ANALYZE
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


-> Filter: (orders.freight > 50.00)  (cost=3940 rows=12997) (actual time=0.126..7.76 rows=40000 loops=1)
-> Table scan on orders  (cost=3940 rows=38996) (actual time=0.123..4.54 rows=40000 loops=1)


---
# Chapter 5 - Sargable filters

## What "sargable" means (simple words)

**Sargable** = the `WHERE` clause is written so MySQL can use an **index on that column** in a clean way.

- Good filters compare the **column itself** to a value or range.
- Awkward filters wrap the column in a **function** first. Then MySQL often cannot jump into the middle of the index.

## Door-lock story

Imagine `order_date` is a locker key hanging on a hook (the index is sorted by that key).

```
Good (sargable)                         Awkward (often weaker)
---------------------------------       ---------------------------------
Use the key as it is                    Wrap the key in thick tape first
order_date >= '2023-01-01'              EXTRACT(YEAR FROM order_date) = 2023
AND order_date <  '2024-01-01'
```

Same business question: "orders in year 2023."  
Different writing style: range on the bare column vs extracting the year.

## Side-by-side meaning

| Style | Filter idea | Index-friendly? |
|-------|-------------|-----------------|
| Date **range** | From 1 Jan 2023 up to (not including) 1 Jan 2024 | Usually yes - index can walk a range |
| `EXTRACT(YEAR ...)` | "Compute year for every row, then keep 2023" | Often weaker - harder to use as a tight range |

## Practical habits (keep these)

1. Prefer filters on columns that already have indexes.
2. Prefer **ranges on the bare column** over `EXTRACT(YEAR FROM col)` / `YEAR(col)` in `WHERE`.
3. Select only the columns you need.
4. Use `LIMIT` while you explore.
5. Join on keys (`customer_id`, `order_id`, ...) - indexes help joins too.

Next: run the same "orders in 2023" question both ways and compare `EXPLAIN` / `EXPLAIN ANALYZE`.


### Good shape: sargable date range

Ask for 2023 using a **from / until** range on `order_date` itself.

Expect something like `type=range` and `key` pointing at the date index (names can vary slightly).


In [7]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys         | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | range | idx_orders_order_date | idx_orders_order_date | 3       | NULL | 19498 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+


### Weaker shape: year extracted in WHERE

Same question ("orders in 2023"), but the filter wraps `order_date` in `EXTRACT(YEAR ...)`.

Compare with the range plan:

| Look at | Range version (above) | Extract version (this cell) |
|---------|------------------------|-------------------------------|
| `type` | often `range` | often wider (`index` / scan-like) |
| `key` | date index used as a range | may still touch the index, but less tightly |
| `rows` | estimate for the year range | often larger / less precise for the same idea |


In [8]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | index | NULL          | idx_orders_order_date | 3       | NULL | 38996 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+


### Same two shapes with EXPLAIN ANALYZE

Now **time** both filters.

- Prefer the **range** form when you need a year filter and have an `order_date` index.
- Read the tree as plain text below (actual time + rows), not only the Chapter 3 table columns.


In [9]:
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


-> Aggregate: count(0)  (cost=5854 rows=1) (actual time=4.97..4.97 rows=1 loops=1)
-> Filter: ((orders.order_date >= DATE'2023-01-01') and (orders.order_date < DATE'2024-01-01'))  (cost=3904 rows=19498) (actual time=0.104..4.15 rows=20905 loops=1)
-> Covering index range scan on orders using idx_orders_order_date over ('2023-01-01' <= order_date < '2024-01-01')  (cost=3904 rows=19498) (actual time=0.101..2.23 rows=20905 loops=1)


In [10]:
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


-> Aggregate: count(0)  (cost=7839 rows=1) (actual time=5.9..5.9 rows=1 loops=1)
-> Filter: (extract(year from orders.order_date) = 2023)  (cost=3940 rows=38996) (actual time=0.143..5.25 rows=20905 loops=1)
-> Covering index scan on orders using idx_orders_order_date  (cost=3940 rows=38996) (actual time=0.141..3.06 rows=40000 loops=1)


### What you should notice

1. Both answers count 2023 orders - the **business question** is the same.
2. The **range** form is the index-friendly (sargable) way to write it.
3. `EXPLAIN` shows the plan shape; `EXPLAIN ANALYZE` shows whether that shape was cheap on your data.

Rule of thumb: **filter on the column, not on a function of the column**, when you want the index to help.


---
# Chapter 6 - Indexes with a TEMPORARY table

## Why this chapter exists

On the shared `training.*` tables we usually **do not** add permanent indexes during class.

Instead we:

1. Copy a **small slice** of orders into a temporary table
2. Add an index on that copy
3. Run `EXPLAIN` / `EXPLAIN ANALYZE` to **feel** the index

A temporary table lives only for this connection. When you disconnect, it disappears. Safe on your laptop.

## Steps in plain words

```
training.orders  --(filter a few months)-->  temp_orders
                                             |
                                             +-- CREATE INDEX on customer_id
                                             |
                                             +-- EXPLAIN lookup WHERE customer_id = 1001
```

## If something goes wrong

> If you re-run the create cell in the **same** session and see "already exists", open a **new** SQL tab / reconnect, then run the cell once.

You can also run `DROP TEMPORARY TABLE IF EXISTS temp_orders;` first if your client allows it in the same script.


### Build temp table, add index, then explain the lookup

This one cell does four things **in the same session** (needed so the temp table still exists):

1. `CREATE TEMPORARY TABLE temp_orders AS ...` - small copy (Q1 2023)
2. `CREATE INDEX ... ON temp_orders (customer_id)` - add the shortcut
3. `EXPLAIN ... WHERE customer_id = 1001` - plan table (`type` / `key` / `rows`)
4. `EXPLAIN ANALYZE ...` - timed tree for the same lookup

What you hope to see on `EXPLAIN`:

| Column | Likely healthy result |
|--------|------------------------|
| `table` | `temp_orders` |
| `type` | `ref` (index lookup) |
| `key` | `temp_orders_customer_id_idx` |
| `rows` | small (near 1 for this customer in the slice) |

Then confirm with `EXPLAIN ANALYZE` that the lookup is cheap.


In [11]:
CREATE TEMPORARY TABLE temp_orders AS
SELECT order_id, customer_id, order_date, status, freight
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2023-04-01';

CREATE INDEX temp_orders_customer_id_idx
  ON temp_orders (customer_id);

EXPLAIN
SELECT *
FROM temp_orders
WHERE customer_id = 1001;

EXPLAIN ANALYZE
SELECT *
FROM temp_orders
WHERE customer_id = 1001;


-- EXPLAIN (plan table fields as one row)
id=1 select_type=SIMPLE table=temp_orders type=ref
possible_keys=temp_orders_customer_id_idx
key=temp_orders_customer_id_idx rows=1 Extra=NULL

-- EXPLAIN ANALYZE (timed tree)
-> Index lookup on temp_orders using temp_orders_customer_id_idx (customer_id=1001)  (cost=0.35 rows=1) (actual time=0.00939..0.0103 rows=1 loops=1)


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Sargable filters</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
